In [ ]:
# 1. Akses Sumber Data

import requests

url = "https://raw.githubusercontent.com/bachtiyarma/FSB/main/Project%20-%20BikeStore/ListDataBikeStore.json"
response = requests.get(url)

if response.status_code == 200:
    list_sheet_id = response.json()
    print(f"Variabel list_sheet_id berhasil diambil dengan tipe data: {type(list_sheet_id)}")
else:
    raise Exception(f"Gagal mengambil metadata JSON. Status code: {response.status_code}")

Variabel list_sheet_id berhasil diambil dengan tipe data: <class 'dict'>


In [ ]:
# 2. Ekstraksi & Metadata Preparation

import pytz
import pandas as pd
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

def extract_data(sheet_id):
    # Menggunakan format export xlsx dari Google Sheets
    url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx'
    data = pd.read_excel(url)

    # Hapus spasi di awal/akhir dan ganti spasi tengah dengan underscore agar BigQuery tidak menolak nama kolom yang mengandung spasi
    data.columns = [str(col).strip().replace(' ', '_').lower() for col in data.columns]

    return data

    # Tambah kolom metadata standar Bronze Layer
    # Memastikan tipe data datetime dipahami oleh Pandas & BigQuery
def add_metadata(data):
    now_utc = datetime.now(timezone.utc)
    date_now = now_utc.astimezone(pytz.timezone('Asia/Jakarta'))

    data_add_md = data.copy()

    data_add_md['job_created_date'] = pd.to_datetime(date_now)
    data_add_md['job_created_by'] = 'SYSTEM'

    return data_add_md

In [ ]:
# 3. Load to Google BigQuery

from google.colab import auth
from google.cloud import bigquery
from pandas_gbq import to_gbq

# Proses autentikasi akun Google Cloud di Colab
auth.authenticate_user()
print('Authenticated Successfully!')

project_id = 'fsb-12345'
dataset_id_brz = 'brz_bike_store'

# Inisialisasi BigQuery Client
client = bigquery.Client(project=project_id)
dataset_ref = bigquery.Dataset(f'{project_id}.{dataset_id_brz}')
dataset_ref.location = "US"
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset '{dataset_id_brz}' siap digunakan di BigQuery.")

def load_data_to_gbq(data, table_name, id_dataset):
    destination = f'{id_dataset}.{table_name}'

    # Mengubah kolom tipe 'object'menjadi string murni, agar BigQuery tidak bingung mendeteksi skema
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col] = data[col].astype(str)

    print(f"Mengunggah {len(data)} baris ke tabel BigQuery: {destination}...")

    # Proses load ke Google BigQuery
    to_gbq(
        data,
        destination_table=destination,
        project_id=project_id,
        if_exists='replace',
        progress_bar=False
    )
    print(f"Sukses memuat tabel {table_name}!")

Authenticated Successfully!
Dataset 'brz_bike_store' siap digunakan di BigQuery.


In [ ]:
# 4. Run Automated Pipeline

print("\n=== Memulai Pipeline ETL Layer Bronze ===")

for key, value in list_sheet_id.items():
    table_name = f"raw_{key}"
    sheet_id_table = value.get('sheet_id')

    if not sheet_id_table:
        print(f"Warning: Key '{key}' tidak memiliki sheet_id. Dilewati.")
        continue

    print(f'\n-> Memproses Tabel: {table_name}')
    print(f'   ID Spreadsheet: {sheet_id_table}')

    try:
        # Step 1: Extract
        data_bronze = extract_data(sheet_id_table)

        # Step 2: Transform (Add Metadata)
        data_bronze_md = add_metadata(data_bronze)

        # Step 3: Load
        load_data_to_gbq(
            data=data_bronze_md,
            table_name=table_name,
            id_dataset=dataset_id_brz
        )

    except Exception as e:
        print(f"❌ Gagal memproses tabel {table_name}. Error: {e}")

print("\n=== Pipeline ETL Layer Bronze Selesai Dijalankan ===")


=== Memulai Pipeline ETL Layer Bronze ===

-> Memproses Tabel: raw_brands
   ID Spreadsheet: 1Ag8PEBWoffpsIIon4xNRUedxrp2bd-b0
Mengunggah 9 baris ke tabel BigQuery: brz_bike_store.raw_brands...
Sukses memuat tabel raw_brands!

-> Memproses Tabel: raw_categories
   ID Spreadsheet: 1-cpMNqgTtdZVBbyCFnil0Nu8qidoVvet
Mengunggah 7 baris ke tabel BigQuery: brz_bike_store.raw_categories...
Sukses memuat tabel raw_categories!

-> Memproses Tabel: raw_customers
   ID Spreadsheet: 1TZKWnnIwseevEfSor4MO5cXip-7UNXRH
Mengunggah 1445 baris ke tabel BigQuery: brz_bike_store.raw_customers...
Sukses memuat tabel raw_customers!

-> Memproses Tabel: raw_order_items
   ID Spreadsheet: 1BRPqL9jA8EHQbLncGRvM5qgFT1DA1HGk
Mengunggah 4722 baris ke tabel BigQuery: brz_bike_store.raw_order_items...
Sukses memuat tabel raw_order_items!

-> Memproses Tabel: raw_orders
   ID Spreadsheet: 1knh9SIiPMhedkxbwF66QICmpxb2L0jBJ
Mengunggah 1615 baris ke tabel BigQuery: brz_bike_store.raw_orders...
Sukses memuat tabel raw_